In [12]:
# Import torch before numpy/pandas in this Windows conda environment.
try:
    import torch
except ModuleNotFoundError:
    torch = None

import numpy as np
import pandas as pd

from src.aergarch_functions import *

set_seed(DEFAULT_SEED)
np.set_printoptions(linewidth=200)
from src.model_sections import *


# SPX Data


In [ ]:
from src.data_loader import get_spx_data

spx_data = get_spx_data()

market_list_2 = spx_data["market_list_2"]
data_SPX = spx_data["data_SPX"]
rms_insample = spx_data["rms_insample"]
rms_outsample = spx_data["rms_outsample"]
returns_insample = spx_data["returns_insample"]
returns_outsample = spx_data["returns_outsample"]

print(f"Loaded {market_list_2[0]} successfully!")
print(f"In-sample observations: {len(rms_insample['SPX'])}")
print(f"Out-of-sample observations: {len(rms_outsample['SPX'])}")


# LW-RealGARCH


In [ ]:
lw_rgarch_spx = run_lw_realgarch_spx(
    market_list_2,
    rms_insample,
    rms_outsample,
    returns_insample,
    returns_outsample,
)



In [ ]:
lw_allparams_df, lw_forecast_nll, lw_synxt_insample_dict = collect_lw_results(
    lw_rgarch_spx,
    rms_insample,
)

f_nll_lw = pd.DataFrame(
    lw_forecast_nll,
    columns=["Market", "LW_RGARCH"],
).set_index("Market")


# FW-RealGARCH


In [ ]:
fw_rgarch_spx = run_fw_realgarch_spx(
    market_list_2,
    rms_insample,
    rms_outsample,
    returns_insample,
    returns_outsample,
)



In [ ]:
fw_allparams_df, fw_forecast_nll, fw_synxt_insample_dict = collect_fw_results(
    fw_rgarch_spx,
    rms_insample,
)

f_nll_fw = pd.DataFrame(
    fw_forecast_nll,
    columns=["Market", "FW_RGARCH"],
).set_index("Market")


# AVG-RealGARCH


In [ ]:
rms_insample_avg, rms_outsample_avg = build_avg_measures(
    market_list_2,
    rms_insample,
    rms_outsample,
)

avg_rgarch_spx = run_avg_realgarch_spx(
    market_list_2,
    rms_insample_avg,
    rms_outsample_avg,
    returns_insample,
    returns_outsample,
)



In [ ]:
avg_allparams_df, avg_forecast_nll = collect_single_measure_results(
    avg_rgarch_spx,
    "AVG",
)

f_nll_avg = pd.DataFrame(
    avg_forecast_nll,
    columns=["Market", "AVG_RGARCH"],
).set_index("Market")


# Realized GARCH (RV5)


In [ ]:
markets_to_run = market_list_2

rv5_rgarch_spx = run_rv5_realgarch_spx(
    markets_to_run,
    rms_insample,
    rms_outsample,
    returns_insample,
    returns_outsample,
)



In [ ]:
rv5_allparams_df, rv5_forecast_nll = collect_single_measure_results(
    rv5_rgarch_spx,
    "RV5",
)

f_nll_rv5 = pd.DataFrame(
    rv5_forecast_nll,
    columns=["Market", "RV5_RGARCH"],
).set_index("Market")


# PCA, ICA


In [ ]:
markets_to_run = market_list_2

pca_results, ica_results = run_pca_ica_spx(
    markets_to_run,
    rms_insample,
    rms_outsample,
)


## PC-RealGARCH


In [ ]:
pca_rgarch_spx = run_pca_realgarch_spx(
    markets_to_run,
    pca_results,
    returns_insample,
    returns_outsample,
)



In [ ]:
pca_allparams_df, pca_forecast_nll = collect_single_measure_results(
    pca_rgarch_spx,
    "PCA",
)

pca_synxt_insample_dict = {}
f_nll_pca = pd.DataFrame(
    pca_forecast_nll,
    columns=["Market", "PCA_RGARCH"],
).set_index("Market")


## IC-RealGARCH


In [ ]:
ica_rgarch_spx = run_ica_realgarch_spx(
    markets_to_run,
    ica_results,
    returns_insample,
    returns_outsample,
)



In [ ]:
ica_allparams_df, ica_forecast_nll = collect_single_measure_results(
    ica_rgarch_spx,
    "ICA",
)

f_nll_ica = pd.DataFrame(
    ica_forecast_nll,
    columns=["Market", "ICA_RGARCH"],
).set_index("Market")


# Autoencoder


In [ ]:
from src.autoencoder_sections import *

print("Using:", device)


## Rolling Autoencoder (12-12-1 SPX Example)


In [ ]:
markets_to_run = market_list_2

AE_ARCHITECTURE = DEFAULT_AE_ARCHITECTURE_12121  # 12-12-1 structure: [12, 12, 1, 12, 12]
AE_HPARAMS = DEFAULT_AE_HPARAMS.copy()

all_market_encoded_series, all_market_weights = run_rolling_ae_spx(
    markets_to_run,
    rms_insample,
    rms_outsample,
    architecture=AE_ARCHITECTURE,
    hparams=AE_HPARAMS,
)


## AE-RealGARCH (12-12-1)


In [ ]:
ae_rgarch_spx = run_ae_realgarch_spx(
    markets_to_run,
    all_market_encoded_series,
    returns_insample,
    returns_outsample,
)



In [ ]:
ae_allparams_df, ae_forecast_nll = collect_ae_results(
    ae_rgarch_spx,
)

f_nll_ae = pd.DataFrame(
    ae_forecast_nll,
    columns=["Market", "AE_RGARCH"],
).set_index("Market")


In [ ]:
covid_start_date = '2020-01-30'
covid_reconised_date = '2020-06-30'
data_rows = []
markets = list(ae_allparams_df.keys())
for m in markets:
    lw_f_nll_covid = lw_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    fw_f_nll_covid = fw_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    avg_f_nll_covid = avg_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    ae_f_nll_covid = ae_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    # ae_f_nll_covid_v2 = ae_allparams_df_v2[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    pca_f_nll_covid = pca_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    ica_f_nll_covid = ica_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    rv5_f_nll_covid = rv5_allparams_df[m].loc[covid_start_date:covid_reconised_date, 'Each_f_nLL'].sum()
    data_rows.append({
        'Market': m,
        'LW': lw_f_nll_covid,
        'FW': fw_f_nll_covid,
        'AVG': avg_f_nll_covid,
        'AE 12121': ae_f_nll_covid,
        # 'AE 1261': ae_f_nll_covid_v2,
        'PCA': pca_f_nll_covid,
        'ICA': ica_f_nll_covid,
        'RV5': rv5_f_nll_covid
    })
df_covid_nll = pd.DataFrame(data_rows)
# df_covid_nll = df_covid_nll[['Market', 'LW', 'FW', 'AVG', 'AE 12121', 'AE 1261', 'PCA', 'ICA', 'RV5']]

# VaR, ES Loss


In [ ]:
from src.risk_sections import *

alpha = DEFAULT_ALPHA
markets_to_run = market_list_2
markets_list = list(ae_allparams_df.keys())

allmodels_params_vares_markets = build_model_data_dict(
    lw_allparams_df,
    fw_allparams_df,
    avg_allparams_df,
    ae_allparams_df,
    pca_allparams_df,
    ica_allparams_df,
    rv5_allparams_df,
)

loaded_data = allmodels_params_vares_markets
model_mapping = DEFAULT_MODEL_MAPPING.copy()
model_keys = DEFAULT_MODEL_KEYS.copy()

add_var_es_to_models(
    loaded_data,
    markets_list,
    alpha=alpha,
)

add_loss_series_to_models(
    loaded_data,
    returns_outsample,
    markets_list,
    model_keys=model_keys,
    alpha=alpha,
)


## Covid Period Loss


In [ ]:
covid_start_date = "2020-01-30"
covid_end_date = "2020-06-30"

covid_qloss_result, covid_jointloss_result = compute_covid_loss_tables(
    loaded_data,
    returns_outsample,
    markets_list,
    model_mapping=model_mapping,
    start_date=covid_start_date,
    end_date=covid_end_date,
    alpha=alpha,
)


# MCS


In [ ]:
mcs_alpha = DEFAULT_MCS_ALPHA

mcs_results = run_mcs_all_markets(
    markets_list,
    loaded_data,
    model_mapping=model_mapping,
    mcs_alpha=mcs_alpha,
)

mcs_qloss_included = mcs_results["mcs_qloss_included"]
mcs_jointloss_included = mcs_results["mcs_jointloss_included"]
mcs_pnll_included = mcs_results["mcs_pnll_included"]

mcs_qloss_results = mcs_results["mcs_qloss_results"]
mcs_jointloss_results = mcs_results["mcs_jointloss_results"]
mcs_pnll_results = mcs_results["mcs_pnll_results"]

df_mcs_qloss = mcs_results["df_mcs_qloss"]
df_mcs_jointloss = mcs_results["df_mcs_jointloss"]
df_mcs_pnll = mcs_results["df_mcs_pnll"]

df_included_qloss = mcs_results["df_included_qloss"]
df_included_jointloss = mcs_results["df_included_jointloss"]
df_included_pnll = mcs_results["df_included_pnll"]


## Covid Period MCS


In [ ]:
covid_mcs_results = run_mcs_all_markets(
    markets_list,
    loaded_data,
    model_mapping=model_mapping,
    mcs_alpha=mcs_alpha,
    start_date=covid_start_date,
    end_date=covid_end_date,
)

covid_mcs_qloss_results = covid_mcs_results["mcs_qloss_results"]
covid_mcs_jointloss_results = covid_mcs_results["mcs_jointloss_results"]
covid_mcs_pnll_results = covid_mcs_results["mcs_pnll_results"]

covid_mcs_qloss_included = covid_mcs_results["mcs_qloss_included"]
covid_mcs_jointloss_included = covid_mcs_results["mcs_jointloss_included"]
covid_mcs_pnll_included = covid_mcs_results["mcs_pnll_included"]

df_covid_mcs_qloss = covid_mcs_results["df_mcs_qloss"]
df_covid_mcs_jointloss = covid_mcs_results["df_mcs_jointloss"]
df_covid_mcs_pnll = covid_mcs_results["df_mcs_pnll"]
